# 🌿 Plant Species — Exploratory Data Analysis

**Author:** [Your Name]  
**Dataset:** Iris (public domain) + simulated field study columns  
**Tools:** Python · pandas · NumPy · matplotlib · scikit-learn

---

## Objective
Perform a full exploratory data analysis on a plant species dataset to understand:
- Species distribution across a field study
- Morphological differences between species (sepal, petal measurements)
- Correlations between plant traits
- Collection patterns by region and month

This mirrors real-world botanical field study analysis.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#FAFAF8',
    'axes.facecolor':   '#FAFAF8',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'grid.linestyle':   '--',
    'font.size':        11,
})

COLORS = ['#2D6A4F', '#52B788', '#95D5B2', '#D8F3DC',
          '#1B4332', '#40916C', '#74C69D', '#B7E4C7']

print('Setup complete ✔')

## 2. Load & Prepare Data

In [ ]:
def load_plant_data():
    """Load Iris dataset and add simulated field study columns."""
    iris = load_iris(as_frame=True)
    df = iris.frame.copy()
    df.columns = ['sepal_length_cm', 'sepal_width_cm',
                  'petal_length_cm', 'petal_width_cm', 'species_code']
    species_map = {0: 'Iris setosa', 1: 'Iris versicolor', 2: 'Iris virginica'}
    df['species'] = df['species_code'].map(species_map)
    np.random.seed(42)
    df['region'] = np.random.choice(
        ['Northern Plains', 'Eastern Ghats', 'Western Coast'],
        size=len(df), p=[0.4, 0.35, 0.25])
    months = ['Jan','Feb','Mar','Apr','May','Jun',
              'Jul','Aug','Sep','Oct','Nov','Dec']
    df['month_collected'] = np.random.choice(months, size=len(df))
    df.drop(columns=['species_code'], inplace=True)
    return df

df = load_plant_data()
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 3. Data Inspection

In [ ]:
print('--- Data types ---')
print(df.dtypes)
print('\n--- Missing values ---')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

In [ ]:
df['species'].value_counts()

## 4. Visualisations

We create 8 plots covering: distribution, morphology, correlation, regional and seasonal patterns.

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Plant Species — Exploratory Data Analysis',
             fontsize=20, fontweight='bold', y=0.98, color='#1B4332')
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

numeric_cols = ['sepal_length_cm','sepal_width_cm','petal_length_cm','petal_width_cm']

# Plot 1 — Species distribution
ax1 = fig.add_subplot(gs[0, 0])
counts = df['species'].value_counts()
bars = ax1.bar(counts.index, counts.values, color=COLORS[:3], edgecolor='white')
ax1.set_title('Species Distribution', fontweight='bold', color='#1B4332')
ax1.set_xlabel('Species'); ax1.set_ylabel('Count')
ax1.set_xticklabels(counts.index, rotation=15, ha='right', fontsize=9)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             str(val), ha='center', fontsize=10, fontweight='bold')

# Plot 2 — Sepal length histogram
ax2 = fig.add_subplot(gs[0, 1])
for i, (sp, grp) in enumerate(df.groupby('species')):
    ax2.hist(grp['sepal_length_cm'], bins=12, alpha=0.7,
             label=sp, color=COLORS[i], edgecolor='white')
ax2.set_title('Sepal Length Distribution', fontweight='bold', color='#1B4332')
ax2.set_xlabel('Sepal Length (cm)'); ax2.set_ylabel('Frequency')
ax2.legend(fontsize=8)

# Plot 3 — Petal scatter
ax3 = fig.add_subplot(gs[0, 2])
for i, (sp, grp) in enumerate(df.groupby('species')):
    ax3.scatter(grp['petal_length_cm'], grp['petal_width_cm'],
                label=sp, color=COLORS[i], alpha=0.75, s=40, edgecolors='white')
ax3.set_title('Petal Length vs Width', fontweight='bold', color='#1B4332')
ax3.set_xlabel('Petal Length (cm)'); ax3.set_ylabel('Petal Width (cm)')
ax3.legend(fontsize=8)

# Plot 4 — Boxplot
ax4 = fig.add_subplot(gs[1, 0])
species_list = df['species'].unique()
data_boxes = [df[df['species']==sp]['sepal_width_cm'].values for sp in species_list]
bp = ax4.boxplot(data_boxes, patch_artist=True,
                 medianprops=dict(color='#1B4332', linewidth=2))
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax4.set_title('Sepal Width by Species', fontweight='bold', color='#1B4332')
ax4.set_ylabel('Sepal Width (cm)')
ax4.set_xticklabels([sp.split()[1] for sp in species_list], rotation=10, fontsize=9)

# Plot 5 — Region bar
ax5 = fig.add_subplot(gs[1, 1])
region_counts = df['region'].value_counts()
ax5.barh(region_counts.index, region_counts.values, color=COLORS[1:4], edgecolor='white')
ax5.set_title('Samples by Region', fontweight='bold', color='#1B4332')
ax5.set_xlabel('Count')
for i, v in enumerate(region_counts.values):
    ax5.text(v+0.3, i, str(v), va='center', fontsize=10, fontweight='bold')

# Plot 6 — Correlation heatmap
ax6 = fig.add_subplot(gs[1, 2])
corr = df[numeric_cols].corr()
short_names = ['Sep.L','Sep.W','Pet.L','Pet.W']
im = ax6.imshow(corr.values, cmap='Greens', vmin=-1, vmax=1)
ax6.set_xticks(range(4)); ax6.set_yticks(range(4))
ax6.set_xticklabels(short_names, fontsize=9)
ax6.set_yticklabels(short_names, fontsize=9)
ax6.set_title('Correlation Heatmap', fontweight='bold', color='#1B4332')
for i in range(4):
    for j in range(4):
        ax6.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center',
                 fontsize=9, color='white' if abs(corr.values[i,j])>0.6 else '#1B4332')
plt.colorbar(im, ax=ax6, shrink=0.8)

# Plot 7 — Monthly trend (stacked bar)
ax7 = fig.add_subplot(gs[2, :2])
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby(['month_collected','species']).size().unstack(fill_value=0).reindex(month_order)
bottom = np.zeros(12)
for i, col in enumerate(monthly.columns):
    ax7.bar(monthly.index, monthly[col], bottom=bottom,
            label=col, color=COLORS[i], alpha=0.85, edgecolor='white')
    bottom += monthly[col].values
ax7.set_title('Collection Activity by Month & Species', fontweight='bold', color='#1B4332')
ax7.set_xlabel('Month'); ax7.set_ylabel('Samples Collected')
ax7.legend(loc='upper right', fontsize=9)

# Plot 8 — Mean measurements grouped bar
ax8 = fig.add_subplot(gs[2, 2])
means = df.groupby('species')[numeric_cols].mean()
x = np.arange(len(numeric_cols)); width = 0.25
for i, (sp, row) in enumerate(means.iterrows()):
    ax8.bar(x+i*width, row.values, width,
            label=sp.split()[1], color=COLORS[i], alpha=0.85, edgecolor='white')
ax8.set_title('Mean Measurements by Species', fontweight='bold', color='#1B4332')
ax8.set_ylabel('cm')
ax8.set_xticks(x+width)
ax8.set_xticklabels(['Sep.L','Sep.W','Pet.L','Pet.W'], fontsize=9)
ax8.legend(fontsize=8)

plt.savefig('plant_species_eda.png', dpi=150, bbox_inches='tight', facecolor='#FAFAF8')
plt.show()
print('✔ Plot saved → plant_species_eda.png')

## 5. Key Findings

In [ ]:
print('=' * 50)
print('KEY FINDINGS')
print('=' * 50)

top_species = df['species'].value_counts().idxmax()
print(f'\n1. Most sampled species : {top_species}')

mean_petal = df.groupby('species')['petal_length_cm'].mean()
longest = mean_petal.idxmax()
print(f'2. Longest petals       : {longest} ({mean_petal[longest]:.2f} cm avg)')

corr_vals = corr.abs().unstack()
corr_vals = corr_vals[corr_vals < 1].sort_values(ascending=False)
bp = corr_vals.index[0]
print(f'3. Strongest correlation: {bp[0]} ↔ {bp[1]} (r = {corr_vals.iloc[0]:.2f})')

print(f'4. Top collection region: {df["region"].value_counts().idxmax()}')
print('\n✔ Analysis complete!')

## 6. Conclusion

This EDA reveals clear morphological separation between the three Iris species:
- **Iris setosa** has distinctly smaller petals, making it easily distinguishable
- **Iris virginica** has the largest petals on average
- **Petal length and petal width** are highly correlated (r ≈ 0.96), suggesting a single growth factor controls both
- Sepal width shows the most overlap between species — least useful for classification

**Next steps:** Build a classification model (Week 3 of this portfolio) to predict species from these measurements.